# Notebook 1: Data acquisition and EDA

## Chicago's Most Dangerous Intersections: A Safety and Equity Analysis

**Team Members:** Rohith Moravaneni, Youngseo Chang, Dunyasha S M Yattogoda, Kyoungmin Lee, Ajay Vishwanatha


## Part I: Problem statement

Where should Chicago put its next traffic-safety dollar? This project looks at whether severe crashes (fatal or incapacitating injury) fall more heavily on some communities than others, from the point of view of a CDOT safety director deciding where to invest.

**Data sources**
- Chicago Traffic Crash tables (Crashes, People, Vehicles) from the Chicago Data Portal, Socrata SODA API
- U.S. Census Bureau ACS 5-Year estimates (2023), tract-level income, population, race, and vehicle access
- Chicago community area boundaries (GeoJSON) from the Chicago Data Portal

**What this notebook does**
- Pulls the three crash tables and the Census tract data through their APIs
- Explores each table with attention to the join keys
- Spatially joins crash coordinates to Census tracts and validates the result
- Pulls the 77 community area polygons used in Notebook 2


## Part II: Data Acquisition

### Acquiring the Chicago traffic crash data and Census income data

In [1]:
import numpy as np 
import pandas as pd
import requests
import io

We begin by pulling the three Chicago Traffic Crash tables (Crashes, People, and Vehicles) from the Chicago Data Portal using the Socrata SODA API. An app token is used to avoid rate limiting during data pulls.

In [24]:
import os
# Our registered app token from the City of Chicago Data Portal

APP_TOKEN = os.environ["SOCRATA_APP_TOKEN"]

#### Crashes Dataset (`85ca-t3if`)

This table is the core dataset. It contains one record per crash incident with details such as date/time, GPS coordinates, severity, contributing factors, and road conditions.


We first test the connection by pulling a single row to confirm the endpoint and token work correctly, and to inspect the schema before pulling the full dataset.

In [6]:
url = 'https://data.cityofchicago.org/resource/85ca-t3if.csv?$limit=1'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)
print(response.text)

200
"crash_record_id","crash_date_est_i","crash_date","posted_speed_limit","traffic_control_device","device_condition","weather_condition","lighting_condition","first_crash_type","trafficway_type","lane_cnt","alignment","roadway_surface_cond","road_defect","report_type","crash_type","intersection_related_i","private_property_i","hit_and_run_i","damage","date_police_notified","prim_contributory_cause","sec_contributory_cause","street_no","street_direction","street_name","beat_of_occurrence","photos_taken_i","statements_taken_i","dooring_i","work_zone_i","work_zone_type","workers_present_i","num_units","most_severe_injury","injuries_total","injuries_fatal","injuries_incapacitating","injuries_non_incapacitating","injuries_reported_not_evident","injuries_no_indication","injuries_unknown","crash_hour","crash_day_of_week","crash_month","latitude","longitude","location"
"c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d0e4aead9658bc8db31581d6de649e6e3344da3e9516a47e69091d3af97e29f02c6de064d9d31b

**Inspecting the columns returned to confirm the fields available**

In [7]:
columns = response.text.split('\n')[0].split(',')

In [8]:
len(columns)

48

In [9]:
print(f'Columns: {columns}')

Columns: ['"crash_record_id"', '"crash_date_est_i"', '"crash_date"', '"posted_speed_limit"', '"traffic_control_device"', '"device_condition"', '"weather_condition"', '"lighting_condition"', '"first_crash_type"', '"trafficway_type"', '"lane_cnt"', '"alignment"', '"roadway_surface_cond"', '"road_defect"', '"report_type"', '"crash_type"', '"intersection_related_i"', '"private_property_i"', '"hit_and_run_i"', '"damage"', '"date_police_notified"', '"prim_contributory_cause"', '"sec_contributory_cause"', '"street_no"', '"street_direction"', '"street_name"', '"beat_of_occurrence"', '"photos_taken_i"', '"statements_taken_i"', '"dooring_i"', '"work_zone_i"', '"work_zone_type"', '"workers_present_i"', '"num_units"', '"most_severe_injury"', '"injuries_total"', '"injuries_fatal"', '"injuries_incapacitating"', '"injuries_non_incapacitating"', '"injuries_reported_not_evident"', '"injuries_no_indication"', '"injuries_unknown"', '"crash_hour"', '"crash_day_of_week"', '"crash_month"', '"latitude"', '"l

The columns and number of columns are as expected.

**Loading the single row response into a dataframe to get a cleaner view of the data.**

In [10]:
test_crashes_df = pd.read_csv(io.StringIO(response.text))

In [11]:
test_crashes_df

,crash_record_id,crash_date_est_i,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,crash_month,latitude,longitude,location
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,NaN,2026-04-18T01:45:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,NOT DIVIDED,...,0,0,3,0,1,7,4,41.764609,-87.683387,POINT (-87.683387053281 41.764608912236)


The single-row dataframe looks good. We now scale up to a 100 row sample before committing to a full pull.

In [12]:
url = 'https://data.cityofchicago.org/resource/85ca-t3if.csv?$limit=100'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)

200


The request is successful. We now load the sample into a dataframe.

In [14]:
sample_crashes_df = pd.read_csv(io.StringIO(response.text))

In [16]:
sample_crashes_df

,crash_record_id,crash_date_est_i,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,crash_month,latitude,longitude,location
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,NaN,2026-04-18T01:45:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,NOT DIVIDED,...,0,0,3,0,1,7,4,41.764609,-87.683387,POINT (-87.683387053281 41.764608912236)
1,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,NaN,2026-04-18T01:17:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,...,0,0,6,0,1,7,4,41.886708,-87.629602,POINT (-87.629601563147 41.886708085892)
2,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,NaN,2026-04-18T01:04:00.000,20,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0,0,1,0,1,7,4,41.969284,-87.718427,POINT (-87.718426836301 41.969284284638)
3,55e705a664cbd0ad3961d701adc8af91e07b95904ea670...,NaN,2026-04-18T00:10:00.000,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0,0,4,0,0,7,4,41.943223,-87.747131,POINT (-87.747130750615 41.943222858294)
4,b4090532587c955b92c35d9475fac5a9aaa2bb5c6d7e27...,NaN,2026-04-17T23:50:00.000,30,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,DIVIDED - W/MEDIAN BARRIER,...,0,0,2,0,23,6,4,41.887056,-87.634937,POINT (-87.634937037709 41.887056468859)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,c28905ff4bbba508f5e1843b39834470fb6c51b5f51c40...,NaN,2026-04-17T16:15:00.000,30,UNKNOWN,UNKNOWN,CLEAR,DAYLIGHT,SIDESWIPE SAME DIRECTION,NOT DIVIDED,...,0,0,2,0,16,6,4,41.757149,-87.576252,POINT (-87.576252405878 41.757148512785)
96,5213e46994a14635e34360c86675afc76f528fd766dfdc...,NaN,2026-04-17T16:13:00.000,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,REAR END,DIVIDED - W/MEDIAN (NOT RAISED),...,2,0,5,0,16,6,4,41.721534,-87.643362,POINT (-87.643361917193 41.721534427876)
97,7c1ed961bacb70a4b9ee187b3e33a8c87fc3a4c4f9e18d...,NaN,2026-04-17T16:10:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,REAR END,NOT DIVIDED,...,0,0,2,0,16,6,4,41.736383,-87.615250,POINT (-87.615250137862 41.73638298642)
98,ebd0ea1ac727c07a9b28f4ca3784ebea42b253f476705e...,NaN,2026-04-17T16:07:00.000,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,ANGLE,FOUR WAY,...,0,1,1,0,16,6,4,41.960679,-87.759977,POINT (-87.759976814017 41.960679195375)


In [17]:
sample_crashes_df['crash_record_id'].isnull().sum()

np.int64(0)

No null values in `crash_record_id`. We are good to proceed with the full data pull.

We now pull the full Crashes dataset using the Socrata v3 `/export.csv` endpoint. Unlike the paginated query endpoint, the export endpoint returns the entire dataset in a single request, so there is no need for pagination, offsets, or rate-limiting delays.

In [18]:
url = 'https://data.cityofchicago.org/api/v3/views/85ca-t3if/export.csv'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)

200


In [20]:
crashes_df = pd.read_csv(io.StringIO(response.text), low_memory=False)

In [21]:
crashes_df

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,NaN,04/18/2026 01:45:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,NOT DIVIDED,...,0.0,0.0,3.0,0.0,1,7,4,41.764609,-87.683387,POINT (-87.683387053281 41.764608912236)
1,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,NaN,04/18/2026 01:17:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,...,0.0,0.0,6.0,0.0,1,7,4,41.886708,-87.629602,POINT (-87.629601563147 41.886708085892)
2,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,NaN,04/18/2026 01:04:00 AM,20,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,1.0,0.0,1,7,4,41.969284,-87.718427,POINT (-87.718426836301 41.969284284638)
3,55e705a664cbd0ad3961d701adc8af91e07b95904ea670...,NaN,04/18/2026 12:10:00 AM,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0.0,0.0,4.0,0.0,0,7,4,41.943223,-87.747131,POINT (-87.747130750615 41.943222858294)
4,b4090532587c955b92c35d9475fac5a9aaa2bb5c6d7e27...,NaN,04/17/2026 11:50:00 PM,30,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,DIVIDED - W/MEDIAN BARRIER,...,0.0,0.0,2.0,0.0,23,6,4,41.887056,-87.634937,POINT (-87.634937037709 41.887056468859)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046564,1d0232afecbdfd01968555aa956a688fd6f55a2bd1984f...,NaN,02/24/2014 07:45:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,DAYLIGHT,TURNING,NOT DIVIDED,...,0.0,0.0,2.0,0.0,19,2,2,41.884016,-87.701143,POINT (-87.701142757538 41.884016475152)
1046565,957783a4787318f005a7dbc920e4c84cb9ac8aa7329a62...,NaN,01/21/2014 07:40:00 AM,30,YIELD,NO CONTROLS,CLEAR,DAYLIGHT,ANGLE,DIVIDED - W/MEDIAN (NOT RAISED),...,1.0,0.0,1.0,0.0,7,3,1,41.760710,-87.561946,POINT (-87.561946030143 41.760710194223)
1046566,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,NaN,01/18/2014 06:14:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,DUSK,PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN BARRIER,...,0.0,0.0,2.0,0.0,18,7,1,41.885610,-87.638756,POINT (-87.638756189808 41.885609917063)
1046567,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,06/01/2013 08:29:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PEDESTRIAN,NOT DIVIDED,...,1.0,0.0,1.0,0.0,20,7,6,42.005145,-87.689969,POINT (-87.689968732521 42.005144534455)


In [40]:
crashes_df['CRASH_RECORD_ID'].isnull().sum()

np.int64(0)

**The full Crashes dataset loaded successfully with over 1 million records and 48 columns.**

Saving the dataframe locally so we don't need to re-pull from the API

In [22]:
crashes_df.to_csv('crashes.csv', index=False)

#### People Dataset (`u6pd-qa9d`)

This table contains one row per person involved in a crash. It includes injury severity, person type (driver/pedestrian), age, and sex. It shares `crash_record_id` with the Crashes table and will be joined during analysis.

**Testing a single row data pull**

In [24]:
url = 'https://data.cityofchicago.org/resource/u6pd-qa9d.csv?$limit=1'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)
print(response.text)

200
"person_id","person_type","crash_record_id","vehicle_id","crash_date","seat_no","city","state","zipcode","sex","age","drivers_license_state","drivers_license_class","safety_equipment","airbag_deployed","ejection","injury_classification","hospital","ems_agency","ems_run_no","driver_action","driver_vision","physical_condition","pedpedal_action","pedpedal_visibility","pedpedal_location","bac_result","bac_result_value","cell_phone_use"
"O2286245","DRIVER","c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d0e4aead9658bc8db31581d6de649e6e3344da3e9516a47e69091d3af97e29f02c6de064d9d31b9a81a","2180389","2026-04-18T01:45:00.000",,"CHICAGO","IL","60652","M","26","IL",,"SAFETY BELT USED","DID NOT DEPLOY","NONE","NO INDICATION OF INJURY",,,,"NONE","NOT OBSCURED","NORMAL",,,,"TEST NOT OFFERED",,



In [25]:
columns = response.text.split('\n')[0].split(',')

In [26]:
len(columns)

29

In [27]:
print(f'Columns: {columns}')

Columns: ['"person_id"', '"person_type"', '"crash_record_id"', '"vehicle_id"', '"crash_date"', '"seat_no"', '"city"', '"state"', '"zipcode"', '"sex"', '"age"', '"drivers_license_state"', '"drivers_license_class"', '"safety_equipment"', '"airbag_deployed"', '"ejection"', '"injury_classification"', '"hospital"', '"ems_agency"', '"ems_run_no"', '"driver_action"', '"driver_vision"', '"physical_condition"', '"pedpedal_action"', '"pedpedal_visibility"', '"pedpedal_location"', '"bac_result"', '"bac_result_value"', '"cell_phone_use"']


The columns and number of columns look good. We now scale up to a 100-row sample before committing to a full pull.

In [28]:
url = 'https://data.cityofchicago.org/resource/u6pd-qa9d.csv?$limit=100'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)

200


In [29]:
sample_people_df = pd.read_csv(io.StringIO(response.text))

In [30]:
sample_people_df

,person_id,person_type,crash_record_id,vehicle_id,crash_date,seat_no,city,state,zipcode,sex,...,ems_run_no,driver_action,driver_vision,physical_condition,pedpedal_action,pedpedal_visibility,pedpedal_location,bac_result,bac_result_value,cell_phone_use
0,O2286245,DRIVER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180389.0,2026-04-18T01:45:00.000,NaN,CHICAGO,IL,60652.0,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,O2286246,DRIVER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180390.0,2026-04-18T01:45:00.000,NaN,CHICAGO,IL,60636.0,F,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,P502775,PASSENGER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180389.0,2026-04-18T01:45:00.000,3.0,NaN,NaN,NaN,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O2286267,DRIVER,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2180407.0,2026-04-18T01:17:00.000,NaN,HIGHLAND PARK,IL,60035.0,M,...,NaN,IMPROPER TURN,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O2286268,DRIVER,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2180417.0,2026-04-18T01:17:00.000,NaN,CHICAGO,IL,60653.0,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,P502724,PASSENGER,05d39d0db5f64ba7bfb112e9ffb7d93feb91e4716a1d12...,2180241.0,2026-04-17T19:15:00.000,1.0,CHICAGO,IL,60641.0,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,O2286068,DRIVER,e0521a4e1ee5b68d1b0e403a1cae5068bd0c4cbb8fef6e...,2180215.0,2026-04-17T19:11:00.000,NaN,CHICAGO,IL,60619.0,F,...,NaN,IMPROPER LANE CHANGE,UNKNOWN,EMOTIONAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
97,O2286069,DRIVER,e0521a4e1ee5b68d1b0e403a1cae5068bd0c4cbb8fef6e...,2180218.0,2026-04-17T19:11:00.000,NaN,CHICAGO,IL,60624.0,F,...,NaN,UNKNOWN,UNKNOWN,EMOTIONAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
98,O2286143,DRIVER,3c5c6d5022be22fa63c928812580f58a02f604f3faed8b...,2180305.0,2026-04-17T19:00:00.000,NaN,ALTOONA,PA,16601.0,M,...,NaN,UNKNOWN,UNKNOWN,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [32]:
sample_people_df[['person_id','crash_record_id']].isnull().sum()

person_id          0
crash_record_id    0
dtype: int64

No null values in `person_id` and `crash_record_id`. We are good to proceed with the full data pull using the export endpoint.

In [33]:
url = 'https://data.cityofchicago.org/api/v3/views/u6pd-qa9d/export.csv'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)

200


In [34]:
people_df = pd.read_csv(io.StringIO(response.text), low_memory=False)

In [35]:
people_df

,PERSON_ID,PERSON_TYPE,CRASH_RECORD_ID,VEHICLE_ID,CRASH_DATE,SEAT_NO,CITY,STATE,ZIPCODE,SEX,...,EMS_RUN_NO,DRIVER_ACTION,DRIVER_VISION,PHYSICAL_CONDITION,PEDPEDAL_ACTION,PEDPEDAL_VISIBILITY,PEDPEDAL_LOCATION,BAC_RESULT,BAC_RESULT VALUE,CELL_PHONE_USE
0,O2286245,DRIVER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180389.0,04/18/2026 01:45:00 AM,NaN,CHICAGO,IL,60652,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,O2286246,DRIVER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180390.0,04/18/2026 01:45:00 AM,NaN,CHICAGO,IL,60636,F,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,P502775,PASSENGER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180389.0,04/18/2026 01:45:00 AM,3.0,NaN,NaN,NaN,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O2286267,DRIVER,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2180407.0,04/18/2026 01:17:00 AM,NaN,HIGHLAND PARK,IL,60035,M,...,NaN,IMPROPER TURN,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O2286268,DRIVER,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2180417.0,04/18/2026 01:17:00 AM,NaN,CHICAGO,IL,60653,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2297073,O1399482,PEDESTRIAN,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,06/01/2013 08:29:00 PM,NaN,CHICAGO,IL,60659,M,...,NaN,UNKNOWN,UNKNOWN,NORMAL,WITH TRAFFIC,NO CONTRASTING CLOTHING,IN ROADWAY,TEST NOT OFFERED,NaN,NaN
2297074,O1399483,DRIVER,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,1329848.0,06/01/2013 08:29:00 PM,NaN,CHICAGO,IL,60645,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2297075,O595730,DRIVER,a802658be15312809c771559e4f81088cfb226830792a5...,567862.0,03/03/2013 04:48:00 PM,NaN,NaN,NaN,NaN,X,...,NaN,FAILED TO YIELD,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2297076,O595731,DRIVER,a802658be15312809c771559e4f81088cfb226830792a5...,567870.0,03/03/2013 04:48:00 PM,NaN,HOFFMAN ESTATES,IL,60169,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [39]:
people_df[['PERSON_ID','CRASH_RECORD_ID']].isnull().sum()

PERSON_ID          0
CRASH_RECORD_ID    0
dtype: int64

**The full People dataset loaded successfully with over 2.2 million records and 29 columns.**

Saving the People dataset locally

In [36]:
people_df.to_csv('people.csv', index=False)

#### Vehicles Dataset (`68nd-jvt3`)
This dataset contains one record per vehicle involved in a crash. It includes vehicle type, maneuver, and defects. It links to the Crashes table via `CRASH_RECORD_ID`.

**Testing a single row data pull**

In [42]:
url = 'https://data.cityofchicago.org/resource/68nd-jvt3.csv?$limit=1'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)
print(response.text)

200
"crash_unit_id","crash_record_id","crash_date","unit_no","unit_type","num_passengers","vehicle_id","cmrc_veh_i","make","model","lic_plate_state","vehicle_year","vehicle_defect","vehicle_type","vehicle_use","travel_direction","maneuver","towed_i","fire_i","occupant_cnt","exceed_speed_limit_i","towed_by","towed_to","area_00_i","area_01_i","area_02_i","area_03_i","area_04_i","area_05_i","area_06_i","area_07_i","area_08_i","area_09_i","area_10_i","area_11_i","area_12_i","area_99_i","first_contact_point","cmv_id","usdot_no","ccmc_no","ilcc_no","commercial_src","gvwr","carrier_name","carrier_state","carrier_city","hazmat_placards_i","hazmat_name","un_no","hazmat_present_i","hazmat_report_i","hazmat_report_no","mcs_report_i","mcs_report_no","hazmat_vio_cause_crash_i","mcs_vio_cause_crash_i","idot_permit_no","wide_load_i","trailer1_width","trailer2_width","trailer1_length","trailer2_length","total_vehicle_length","axle_cnt","vehicle_config","cargo_body_type","load_type","hazmat_out_of_serv

In [43]:
columns = response.text.split('\n')[0].split(',')

In [44]:
len(columns)

71

In [45]:
print(f'Columns: {columns}')

Columns: ['"crash_unit_id"', '"crash_record_id"', '"crash_date"', '"unit_no"', '"unit_type"', '"num_passengers"', '"vehicle_id"', '"cmrc_veh_i"', '"make"', '"model"', '"lic_plate_state"', '"vehicle_year"', '"vehicle_defect"', '"vehicle_type"', '"vehicle_use"', '"travel_direction"', '"maneuver"', '"towed_i"', '"fire_i"', '"occupant_cnt"', '"exceed_speed_limit_i"', '"towed_by"', '"towed_to"', '"area_00_i"', '"area_01_i"', '"area_02_i"', '"area_03_i"', '"area_04_i"', '"area_05_i"', '"area_06_i"', '"area_07_i"', '"area_08_i"', '"area_09_i"', '"area_10_i"', '"area_11_i"', '"area_12_i"', '"area_99_i"', '"first_contact_point"', '"cmv_id"', '"usdot_no"', '"ccmc_no"', '"ilcc_no"', '"commercial_src"', '"gvwr"', '"carrier_name"', '"carrier_state"', '"carrier_city"', '"hazmat_placards_i"', '"hazmat_name"', '"un_no"', '"hazmat_present_i"', '"hazmat_report_i"', '"hazmat_report_no"', '"mcs_report_i"', '"mcs_report_no"', '"hazmat_vio_cause_crash_i"', '"mcs_vio_cause_crash_i"', '"idot_permit_no"', '"wi

The columns and number of columns look good. We now scale up to a 100-row sample before committing to a full pull.

In [46]:
url = 'https://data.cityofchicago.org/resource/68nd-jvt3.csv?$limit=100'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)

200


The request is successful. We now load the sample into a dataframe.

In [47]:
sample_vehicles_df = pd.read_csv(io.StringIO(response.text))

In [49]:
sample_vehicles_df

,crash_unit_id,crash_record_id,crash_date,unit_no,unit_type,num_passengers,vehicle_id,cmrc_veh_i,make,model,...,trailer1_length,trailer2_length,total_vehicle_length,axle_cnt,vehicle_config,cargo_body_type,load_type,hazmat_out_of_service_i,mcs_out_of_service_i,hazmat_class
0,2286245,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2026-04-18T01:45:00.000,1,DRIVER,1.0,2180389.0,NaN,NISSAN,MAXIMA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2286246,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2026-04-18T01:45:00.000,2,DRIVER,NaN,2180390.0,NaN,DODGE,JOURNEY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2286267,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2026-04-18T01:17:00.000,1,DRIVER,1.0,2180407.0,NaN,TOYOTA,RAV4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2286268,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2026-04-18T01:17:00.000,2,DRIVER,3.0,2180417.0,NaN,BUICK,VERANO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2286241,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,2026-04-18T01:04:00.000,1,DRIVER,NaN,2180391.0,NaN,CHEVROLET,CRUZE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2286090,7a41aef2c9377a02a68cdbbebab677a84af3e684cc3cbd...,2026-04-17T18:20:00.000,1,DRIVER,NaN,2180253.0,NaN,ALFA ROMEO,STELVIO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,2286091,7a41aef2c9377a02a68cdbbebab677a84af3e684cc3cbd...,2026-04-17T18:20:00.000,2,PARKED,NaN,2180256.0,NaN,GENERAL MOTORS CORPORATION (GMC),ACADIA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,2286101,237320c9de2ceabc63ee3c06ef6e0c5467c695cd3b22e9...,2026-04-17T18:19:00.000,1,DRIVER,1.0,2180249.0,NaN,FORD,EXPLORER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,2286102,237320c9de2ceabc63ee3c06ef6e0c5467c695cd3b22e9...,2026-04-17T18:19:00.000,2,DRIVER,NaN,2180252.0,NaN,UNKNOWN,OTHER (EXPLAIN IN NARRATIVE),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
sample_vehicles_df['crash_record_id'].isnull().sum()

np.int64(0)

No null values in the join key: `crash_record_id`. We are good to proceed with the full data pull using the export endpoint.

**Pulling the full dataset using the export endpoint**

In [53]:
url = 'https://data.cityofchicago.org/api/v3/views/68nd-jvt3/export.csv'
headers = {'X-App-Token': APP_TOKEN}

response = requests.get(url, headers=headers)
response.raise_for_status()

print(response.status_code)

200


In [54]:
vehicles_df = pd.read_csv(io.StringIO(response.text), low_memory=False)

In [55]:
vehicles_df

,CRASH_UNIT_ID,CRASH_RECORD_ID,CRASH_DATE,UNIT_NO,UNIT_TYPE,NUM_PASSENGERS,VEHICLE_ID,CMRC_VEH_I,MAKE,MODEL,...,TRAILER1_LENGTH,TRAILER2_LENGTH,TOTAL_VEHICLE_LENGTH,AXLE_CNT,VEHICLE_CONFIG,CARGO_BODY_TYPE,LOAD_TYPE,HAZMAT_OUT_OF_SERVICE_I,MCS_OUT_OF_SERVICE_I,HAZMAT_CLASS
0,2286245,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,04/18/2026 01:45:00 AM,1,DRIVER,1.0,2180389.0,NaN,NISSAN,MAXIMA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2286246,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,04/18/2026 01:45:00 AM,2,DRIVER,NaN,2180390.0,NaN,DODGE,JOURNEY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2286267,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,04/18/2026 01:17:00 AM,1,DRIVER,1.0,2180407.0,NaN,TOYOTA,RAV4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2286268,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,04/18/2026 01:17:00 AM,2,DRIVER,3.0,2180417.0,NaN,BUICK,VERANO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2286241,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,04/18/2026 01:04:00 AM,1,DRIVER,NaN,2180391.0,NaN,CHEVROLET,CRUZE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2133829,481322,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,01/18/2014 06:14:00 PM,2,PARKED,1.0,460661.0,NaN,DODGE,CHARGER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2133830,1399482,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,06/01/2013 08:29:00 PM,1,PEDESTRIAN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2133831,1399483,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,06/01/2013 08:29:00 PM,2,DRIVER,NaN,1329848.0,NaN,HONDA,CIVIC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2133832,595730,a802658be15312809c771559e4f81088cfb226830792a5...,03/03/2013 04:48:00 PM,1,DRIVER,NaN,567862.0,NaN,KIA MOTORS CORP,UNKNOWN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
vehicles_df[['CRASH_RECORD_ID', 'VEHICLE_ID']].isnull().sum()

CRASH_RECORD_ID        0
VEHICLE_ID         50814
dtype: int64

No nulls in `CRASH_RECORD_ID`, which is the primary join key. We will investigate the null `VEHICLE_ID` values further during EDA.

**The full Vehicles dataset loaded successfully with over 2.1 million rows and 71 columns.**

Saving the Vehicles dataset locally

In [59]:
vehicles_df.to_csv('vehicles.csv', index=False)

#### Census ACS Data (Tract-level demographics)
We pull tract-level demographic data for Cook County, IL (which contains Chicago) from the U.S. Census Bureau's American Community Survey (ACS) 5-Year estimates API. We acquire six variables that support our crash-rate-per-capita analysis and equity findings.

| Variable | Concept | What we use it for |
|---|---|---|
| `B19013_001E` | Median household income (2023 dollars) | Income-based equity analysis |
| `B01003_001E` | Total population | Denominator for crash rates per 10,000 residents |
| `B03002_001E` | Total population (Hispanic-or-Latino-origin universe) | Denominator for % POC |
| `B03002_003E` | Non-Hispanic white alone population | Numerator; complement gives % POC |
| `B08201_001E` | Total households | Denominator for vehicle access rate |
| `B08201_002E` | Households with no vehicle available | Pedestrian/transit exposure proxy |

Per-variable metadata is available at `https://api.census.gov/data/2023/acs/acs5/variables/{CODE}.html` for each code above.

In [2]:
import os
CENSUS_API_KEY = os.environ["CENSUS_API_KEY"]

Testing the Census API connection by pulling median household income (`B19013_001E`) for a single tract in Cook County, IL.

In [4]:
# UPDATED CELL

base_url = "https://api.census.gov/data/2023/acs/acs5"
params = {
    "get": "B19013_001E,B01003_001E,B03002_001E,B03002_003E,B08201_001E,B08201_002E,NAME",
    "for": "tract:010100",          # single tract for testing
    "in": "state:17 county:031",    # 17 = Illinois, 031 = Cook County
    "key": CENSUS_API_KEY
}

response = requests.get(base_url, params=params)
print(response.status_code)
print(response.json())

200
[['B19013_001E', 'B01003_001E', 'B03002_001E', 'B03002_003E', 'B08201_001E', 'B08201_002E', 'NAME', 'state', 'county', 'tract'], ['69460', '3726', '3726', '1297', '2190', '820', 'Census Tract 101; Cook County; Illinois', '17', '031', '010100']]


The test call returns a 200 with the expected structure. We now pull median household income for all tracts in Cook County.

In [5]:
# UPDATED CELL

base_url = "https://api.census.gov/data/2023/acs/acs5"
params = {
    "get": "B19013_001E,B01003_001E,B03002_001E,B03002_003E,B08201_001E,B08201_002E,NAME",
    "for": "tract:*",                 # all tracts
    "in": "state:17 county:031",      
    "key": CENSUS_API_KEY
}

response = requests.get(base_url, params=params)
print(response.status_code)

200


In [ ]:
response.json()

[['B19013_001E',
  'B01003_001E',
  'B03002_001E',
  'B03002_003E',
  'B08201_001E',
  'B08201_002E',
  'NAME',
  'state',
  'county',
  'tract'],
 ['69460',
  '3726',
  '3726',
  '1297',
  '2190',
  '820',
  'Census Tract 101; Cook County; Illinois',
  '17',
  '031',
  '010100'],
 ['49639',
  '7588',
  '7588',
  '1406',
  '3038',
  '942',
  'Census Tract 102.01; Cook County; Illinois',
  '17',
  '031',
  '010201'],
 ['55119',
  '2609',
  '2609',
  '967',
  '1130',
  '352',
  'Census Tract 102.02; Cook County; Illinois',
  '17',
  '031',
  '010202'],
 ['65871',
  '6311',
  '6311',
  '3094',
  '3185',
  '1326',
  'Census Tract 103; Cook County; Illinois',
  '17',
  '031',
  '010300'],
 ['49017',
  '4282',
  '4282',
  '3173',
  '2058',
  '980',
  'Census Tract 104; Cook County; Illinois',
  '17',
  '031',
  '010400'],
 ['48250',
  '3519',
  '3519',
  '2217',
  '2379',
  '1268',
  'Census Tract 105.01; Cook County; Illinois',
  '17',
  '031',
  '010501'],
 ['61367',
  '3329',
  '3329',
  

The request is successful. We now load the response into a dataframe.

In [7]:
# UPDATED CELL

data = response.json()
census_df = pd.DataFrame(data[1:], columns=data[0])

In [8]:
census_df

,B19013_001E,B01003_001E,B03002_001E,B03002_003E,B08201_001E,B08201_002E,NAME,state,county,tract
0,69460,3726,3726,1297,2190,820,Census Tract 101; Cook County; Illinois,17,031,010100
1,49639,7588,7588,1406,3038,942,Census Tract 102.01; Cook County; Illinois,17,031,010201
2,55119,2609,2609,967,1130,352,Census Tract 102.02; Cook County; Illinois,17,031,010202
3,65871,6311,6311,3094,3185,1326,Census Tract 103; Cook County; Illinois,17,031,010300
4,49017,4282,4282,3173,2058,980,Census Tract 104; Cook County; Illinois,17,031,010400
...,...,...,...,...,...,...,...,...,...,...
1327,56848,2274,2274,114,851,157,Census Tract 8446; Cook County; Illinois,17,031,844600
1328,69583,1906,1906,33,683,112,Census Tract 8447; Cook County; Illinois,17,031,844700
1329,-666666666,0,0,0,0,0,Census Tract 9800; Cook County; Illinois,17,031,980000
1330,-666666666,0,0,0,0,0,Census Tract 9801; Cook County; Illinois,17,031,980100


In [9]:
census_df['tract'].nunique()

1332

1,332 tracts loaded. We note that some tracts show `-666666666` median income. We will investigate this during the EDA. 

In [10]:
# Renaming the variable code columns 

census_df.rename(columns={
    'B19013_001E': 'median_household_income',
    'B01003_001E': 'total_population',
    'B03002_001E': 'race_universe_population',
    'B03002_003E': 'nh_white_population',
    'B08201_001E': 'total_households',
    'B08201_002E': 'zero_vehicle_households',
}, inplace=True)

In [11]:
census_df.head(1)

,median_household_income,total_population,race_universe_population,nh_white_population,total_households,zero_vehicle_households,NAME,state,county,tract
0,69460,3726,3726,1297,2190,820,Census Tract 101; Cook County; Illinois,17,031,010100


Saving the Census data locally

In [12]:
census_df.to_csv('census.csv', index=False)

#### Acquisition summary
We successfully acquired four datasets:
- **Crashes** (`85ca-t3if`): 1,046,569 rows × 48 columns
- **People** (`u6pd-qa9d`): 2,297,078 rows × 29 columns
- **Vehicles** (`68nd-jvt3`): 2,133,834 rows × 71 columns 
- **Census ACS**: 1,332 tracts × 10 columns (six demographic variables: median household income, total population, race composition, and household vehicle access by tract for Cook County, IL)

All three crash tables were pulled via the Socrata SODA API export endpoint, and Census data was pulled via the Census Bureau ACS 5-Year API. All datasets have been saved locally as CSV files.

### Exploratory data analysis

We perform EDA on each dataset, focusing on fields most relevant to our analysis and the join keys that will link the datasets together. We have already confirmed during data acquisition that `CRASH_RECORD_ID` has no null values across all three crash tables.

In [25]:
crashes_df = pd.read_csv('crashes.csv', low_memory=False)
people_df = pd.read_csv('people.csv', low_memory=False)
vehicles_df = pd.read_csv('vehicles.csv', low_memory=False)
census_df = pd.read_csv('census.csv', low_memory=False)

**Crashes EDA**

In [26]:
crashes_df

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,NaN,04/18/2026 01:45:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,NOT DIVIDED,...,0.0,0.0,3.0,0.0,1,7,4,41.764609,-87.683387,POINT (-87.683387053281 41.764608912236)
1,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,NaN,04/18/2026 01:17:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,...,0.0,0.0,6.0,0.0,1,7,4,41.886708,-87.629602,POINT (-87.629601563147 41.886708085892)
2,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,NaN,04/18/2026 01:04:00 AM,20,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,1.0,0.0,1,7,4,41.969284,-87.718427,POINT (-87.718426836301 41.969284284638)
3,55e705a664cbd0ad3961d701adc8af91e07b95904ea670...,NaN,04/18/2026 12:10:00 AM,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0.0,0.0,4.0,0.0,0,7,4,41.943223,-87.747131,POINT (-87.747130750615 41.943222858294)
4,b4090532587c955b92c35d9475fac5a9aaa2bb5c6d7e27...,NaN,04/17/2026 11:50:00 PM,30,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,DIVIDED - W/MEDIAN BARRIER,...,0.0,0.0,2.0,0.0,23,6,4,41.887056,-87.634937,POINT (-87.634937037709 41.887056468859)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046564,1d0232afecbdfd01968555aa956a688fd6f55a2bd1984f...,NaN,02/24/2014 07:45:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,DAYLIGHT,TURNING,NOT DIVIDED,...,0.0,0.0,2.0,0.0,19,2,2,41.884016,-87.701143,POINT (-87.701142757538 41.884016475152)
1046565,957783a4787318f005a7dbc920e4c84cb9ac8aa7329a62...,NaN,01/21/2014 07:40:00 AM,30,YIELD,NO CONTROLS,CLEAR,DAYLIGHT,ANGLE,DIVIDED - W/MEDIAN (NOT RAISED),...,1.0,0.0,1.0,0.0,7,3,1,41.760710,-87.561946,POINT (-87.561946030143 41.760710194223)
1046566,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,NaN,01/18/2014 06:14:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,DUSK,PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN BARRIER,...,0.0,0.0,2.0,0.0,18,7,1,41.885610,-87.638756,POINT (-87.638756189808 41.885609917063)
1046567,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,06/01/2013 08:29:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PEDESTRIAN,NOT DIVIDED,...,1.0,0.0,1.0,0.0,20,7,6,42.005145,-87.689969,POINT (-87.689968732521 42.005144534455)


In [5]:
crashes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1046569 entries, 0 to 1046568
Data columns (total 48 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   CRASH_RECORD_ID                1046569 non-null  object 
 1   CRASH_DATE_EST_I               75797 non-null    object 
 2   CRASH_DATE                     1046569 non-null  object 
 3   POSTED_SPEED_LIMIT             1046569 non-null  int64  
 4   TRAFFIC_CONTROL_DEVICE         1046569 non-null  object 
 5   DEVICE_CONDITION               1046569 non-null  object 
 6   WEATHER_CONDITION              1046569 non-null  object 
 7   LIGHTING_CONDITION             1046569 non-null  object 
 8   FIRST_CRASH_TYPE               1046569 non-null  object 
 9   TRAFFICWAY_TYPE                1046569 non-null  object 
 10  LANE_CNT                       199042 non-null   object 
 11  ALIGNMENT                      1046569 non-null  object 
 12  ROADWAY_SURFAC

**Changing `CRASH_DATE` to datetime**

In [27]:
crashes_df['CRASH_DATE'] = pd.to_datetime(crashes_df['CRASH_DATE'])

**Checking for null values**

In [7]:
crashes_df.isna().sum()

CRASH_RECORD_ID                        0
CRASH_DATE_EST_I                  970772
CRASH_DATE                             0
POSTED_SPEED_LIMIT                     0
TRAFFIC_CONTROL_DEVICE                 0
DEVICE_CONDITION                       0
WEATHER_CONDITION                      0
LIGHTING_CONDITION                     0
FIRST_CRASH_TYPE                       0
TRAFFICWAY_TYPE                        0
LANE_CNT                          847527
ALIGNMENT                              0
ROADWAY_SURFACE_COND                   0
ROAD_DEFECT                            0
REPORT_TYPE                        35087
CRASH_TYPE                             0
INTERSECTION_RELATED_I            805798
NOT_RIGHT_OF_WAY_I                999791
HIT_AND_RUN_I                     718302
DAMAGE                                 0
DATE_POLICE_NOTIFIED                   0
PRIM_CONTRIBUTORY_CAUSE                0
SEC_CONTRIBUTORY_CAUSE                 0
STREET_NO                              0
STREET_DIRECTION

Most columns are fully populated. Key observations:
- `CRASH_RECORD_ID` and `CRASH_DATE` have zero nulls. The primary join key and date field are fully intact.
- `LATITUDE`/`LONGITUDE`: 8030 missing values (about 0.8% of data). These records cannot be used in spatial joins but represent a small fraction of the dataset. This will need to be addressed during the spatial analysis.

**Looking at the distribution of crash severity**

In [12]:
crashes_df['MOST_SEVERE_INJURY'].value_counts(normalize=True)

MOST_SEVERE_INJURY
NO INDICATION OF INJURY     0.856318
NONINCAPACITATING INJURY    0.079966
REPORTED, NOT EVIDENT       0.046457
INCAPACITATING INJURY       0.016209
FATAL                       0.001050
Name: proportion, dtype: float64

The vast majority of crashes (~85%) have no indication of injury. Fatal crashes make up about 0.1% of the dataset

**Examining the top primary contributing causes to understand what factors dominate at crash locations.**

In [110]:
crashes_df['PRIM_CONTRIBUTORY_CAUSE'].value_counts(normalize=True).head(10)

PRIM_CONTRIBUTORY_CAUSE
UNABLE TO DETERMINE                       0.394119
FAILING TO YIELD RIGHT-OF-WAY             0.111540
FOLLOWING TOO CLOSELY                     0.095503
NOT APPLICABLE                            0.051656
IMPROPER OVERTAKING/PASSING               0.050451
FAILING TO REDUCE SPEED TO AVOID CRASH    0.041888
IMPROPER BACKING                          0.037503
IMPROPER LANE USAGE                       0.035274
DRIVING SKILLS/KNOWLEDGE/EXPERIENCE       0.034903
IMPROPER TURNING/NO SIGNAL                0.033573
Name: proportion, dtype: float64

"Unable to determine" is the most common cause, accounting for nearly 40% of crashes. Among identifiable causes, failing to yield right-of-way and following too closely are the top contributors to crashes.

**Confirming the time span of the crash data**

In [15]:
crashes_df['CRASH_DATE'].min(), crashes_df['CRASH_DATE'].max()

(Timestamp('2013-03-03 16:48:00'), Timestamp('2026-04-18 01:45:00'))

In [16]:
crashes_df['CRASH_DATE'].dt.year.value_counts().sort_index()

CRASH_DATE
2013         2
2014         6
2015      9834
2016     44297
2017     83786
2018    118952
2019    117764
2020     92096
2021    108767
2022    108412
2023    110755
2024    112055
2025    109111
2026     30732
Name: count, dtype: int64

The data spans from 2013 to present (April 2026). From the dataset documentation, E-Crash data is only available for some police districts starting in 2015, with full citywide coverage beginning September 2017. Records before 2017 are sparse and incomplete. 2017 and 2026 are partial years. We will consider filtering to complete years (2018-2025) during later analysis.

**Verifying that the crash coordinates fall within Chicago's geographic bounds**

In [19]:
crashes_df[crashes_df['LATITUDE'].notna()][['LATITUDE', 'LONGITUDE']].describe()

,LATITUDE,LONGITUDE
count,1.038539e+06,1.038539e+06
mean,4.185520e+01,-8.767311e+01
std,3.566508e-01,7.273341e-01
min,0.000000e+00,-8.793968e+01
25%,4.178421e+01,-8.772182e+01
50%,4.187517e+01,-8.767445e+01
75%,4.192481e+01,-8.763386e+01
max,4.202278e+01,0.000000e+00


The mean latitude (41.86) and longitude (87.67) align with Chicago's known coordinates (41.85, -87.65). The interquartile ranges also fall within expected city bounds.

We see min latitude of 0.0 and max longitude of 0.0 in the describe output, indicating some records have invalid zero coordinates. Checking how many exist

In [26]:
((crashes_df['LATITUDE'] == 0) | (crashes_df['LONGITUDE'] == 0)).sum()

np.int64(71)

71 records have zero-value coordinates. We replace these with NaN so they are treated as missing alongside the existing 8,030 null values, giving us 8,101 total missing coordinates.

In [28]:
crashes_df.loc[(crashes_df['LATITUDE'] == 0) | (crashes_df['LONGITUDE'] == 0), ['LATITUDE', 'LONGITUDE']] = np.nan

In [29]:
crashes_df[['LATITUDE', 'LONGITUDE']].isnull().sum()

LATITUDE     8101
LONGITUDE    8101
dtype: int64

**People EDA**

We check for missing values in the join keys and examine the person type distribution.

In [30]:
people_df

,PERSON_ID,PERSON_TYPE,CRASH_RECORD_ID,VEHICLE_ID,CRASH_DATE,SEAT_NO,CITY,STATE,ZIPCODE,SEX,...,EMS_RUN_NO,DRIVER_ACTION,DRIVER_VISION,PHYSICAL_CONDITION,PEDPEDAL_ACTION,PEDPEDAL_VISIBILITY,PEDPEDAL_LOCATION,BAC_RESULT,BAC_RESULT VALUE,CELL_PHONE_USE
0,O2286245,DRIVER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180389.0,04/18/2026 01:45:00 AM,NaN,CHICAGO,IL,60652,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,O2286246,DRIVER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180390.0,04/18/2026 01:45:00 AM,NaN,CHICAGO,IL,60636,F,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,P502775,PASSENGER,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,2180389.0,04/18/2026 01:45:00 AM,3.0,NaN,NaN,NaN,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O2286267,DRIVER,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2180407.0,04/18/2026 01:17:00 AM,NaN,HIGHLAND PARK,IL,60035,M,...,NaN,IMPROPER TURN,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O2286268,DRIVER,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,2180417.0,04/18/2026 01:17:00 AM,NaN,CHICAGO,IL,60653,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2297073,O1399482,PEDESTRIAN,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,06/01/2013 08:29:00 PM,NaN,CHICAGO,IL,60659,M,...,NaN,UNKNOWN,UNKNOWN,NORMAL,WITH TRAFFIC,NO CONTRASTING CLOTHING,IN ROADWAY,TEST NOT OFFERED,NaN,NaN
2297074,O1399483,DRIVER,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,1329848.0,06/01/2013 08:29:00 PM,NaN,CHICAGO,IL,60645,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2297075,O595730,DRIVER,a802658be15312809c771559e4f81088cfb226830792a5...,567862.0,03/03/2013 04:48:00 PM,NaN,NaN,NaN,NaN,X,...,NaN,FAILED TO YIELD,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2297076,O595731,DRIVER,a802658be15312809c771559e4f81088cfb226830792a5...,567870.0,03/03/2013 04:48:00 PM,NaN,HOFFMAN ESTATES,IL,60169,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [7]:
people_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2297078 entries, 0 to 2297077
Data columns (total 29 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   PERSON_ID              object 
 1   PERSON_TYPE            object 
 2   CRASH_RECORD_ID        object 
 3   VEHICLE_ID             float64
 4   CRASH_DATE             object 
 5   SEAT_NO                float64
 6   CITY                   object 
 7   STATE                  object 
 8   ZIPCODE                object 
 9   SEX                    object 
 10  AGE                    float64
 11  DRIVERS_LICENSE_STATE  object 
 12  DRIVERS_LICENSE_CLASS  object 
 13  SAFETY_EQUIPMENT       object 
 14  AIRBAG_DEPLOYED        object 
 15  EJECTION               object 
 16  INJURY_CLASSIFICATION  object 
 17  HOSPITAL               object 
 18  EMS_AGENCY             object 
 19  EMS_RUN_NO             object 
 20  DRIVER_ACTION          object 
 21  DRIVER_VISION          object 
 22  PHYSICAL_CONDITION

In [ ]:
people_df[['CRASH_RECORD_ID', 'PERSON_ID']].isnull().sum()

CRASH_RECORD_ID    0
PERSON_ID          0
dtype: int64

No nulls in the join key (`CRASH_RECORD_ID`) or the table's unique identifier (`PERSON_ID`). We now look at the distribution of person types involved in crashes.

In [29]:
people_df['PERSON_TYPE'].value_counts(normalize=True)

PERSON_TYPE
DRIVER                 0.777917
PASSENGER              0.201326
PEDESTRIAN             0.012031
BICYCLE                0.007692
NON-MOTOR VEHICLE      0.000874
NON-CONTACT VEHICLE    0.000160
Name: proportion, dtype: float64

As expected, the majority of people involved in crashes are drivers and passengers.

**Vehicles EDA**

We check the join key for nulls and examine the `UNIT_TYPE` distribution to explain the ~50,000 null `VEHICLE_ID` values we flagged during data acquisition.

In [4]:
vehicles_df

,CRASH_UNIT_ID,CRASH_RECORD_ID,CRASH_DATE,UNIT_NO,UNIT_TYPE,NUM_PASSENGERS,VEHICLE_ID,CMRC_VEH_I,MAKE,MODEL,...,TRAILER1_LENGTH,TRAILER2_LENGTH,TOTAL_VEHICLE_LENGTH,AXLE_CNT,VEHICLE_CONFIG,CARGO_BODY_TYPE,LOAD_TYPE,HAZMAT_OUT_OF_SERVICE_I,MCS_OUT_OF_SERVICE_I,HAZMAT_CLASS
0,2286245,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,04/18/2026 01:45:00 AM,1,DRIVER,1.0,2180389.0,NaN,NISSAN,MAXIMA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2286246,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,04/18/2026 01:45:00 AM,2,DRIVER,NaN,2180390.0,NaN,DODGE,JOURNEY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2286267,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,04/18/2026 01:17:00 AM,1,DRIVER,1.0,2180407.0,NaN,TOYOTA,RAV4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2286268,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,04/18/2026 01:17:00 AM,2,DRIVER,3.0,2180417.0,NaN,BUICK,VERANO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2286241,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,04/18/2026 01:04:00 AM,1,DRIVER,NaN,2180391.0,NaN,CHEVROLET,CRUZE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2133829,481322,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,01/18/2014 06:14:00 PM,2,PARKED,1.0,460661.0,NaN,DODGE,CHARGER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2133830,1399482,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,06/01/2013 08:29:00 PM,1,PEDESTRIAN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2133831,1399483,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,06/01/2013 08:29:00 PM,2,DRIVER,NaN,1329848.0,NaN,HONDA,CIVIC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2133832,595730,a802658be15312809c771559e4f81088cfb226830792a5...,03/03/2013 04:48:00 PM,1,DRIVER,NaN,567862.0,NaN,KIA MOTORS CORP,UNKNOWN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
vehicles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2133834 entries, 0 to 2133833
Data columns (total 71 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   CRASH_UNIT_ID             int64  
 1   CRASH_RECORD_ID           object 
 2   CRASH_DATE                object 
 3   UNIT_NO                   object 
 4   UNIT_TYPE                 object 
 5   NUM_PASSENGERS            float64
 6   VEHICLE_ID                float64
 7   CMRC_VEH_I                object 
 8   MAKE                      object 
 9   MODEL                     object 
 10  LIC_PLATE_STATE           object 
 11  VEHICLE_YEAR              float64
 12  VEHICLE_DEFECT            object 
 13  VEHICLE_TYPE              object 
 14  VEHICLE_USE               object 
 15  TRAVEL_DIRECTION          object 
 16  MANEUVER                  object 
 17  TOWED_I                   object 
 18  FIRE_I                    object 
 19  OCCUPANT_CNT              float64
 20  EXCEED_SPEED_LIMIT_I    

In [31]:
vehicles_df[['CRASH_RECORD_ID', 'VEHICLE_ID']].isnull().sum()

CRASH_RECORD_ID        0
VEHICLE_ID         50814
dtype: int64

No nulls in `CRASH_RECORD_ID`. The ~50,000 null `VEHICLE_ID` values remain. We now check `UNIT_TYPE` for those records to confirm whether they correspond to pedestrians and cyclists.

In [32]:
vehicles_df[vehicles_df['VEHICLE_ID'].isnull()]['UNIT_TYPE'].value_counts()

UNIT_TYPE
PEDESTRIAN             27637
BICYCLE                17668
NON-MOTOR VEHICLE       2008
DRIVER                   729
NON-CONTACT VEHICLE      165
PARKED                   137
DRIVERLESS                31
EQUESTRIAN                 9
DISABLED VEHICLE           4
Name: count, dtype: int64

In [33]:
vehicles_df[vehicles_df['VEHICLE_ID'].isnull()]['UNIT_TYPE'].isnull().sum()

np.int64(2426)

Most of the null `VEHICLE_ID` values are explained by pedestrians (28k) and cyclists (18k), which naturally don't have a vehicle. Combined with the other non-vehicle unit types (27,637 + 17,668 + 2,008 + 729 + 165 + 137 + 31 + 9 + 4 = 48,388), that accounts for 48,388 of the 50,814 nulls. The remaining 2,426 have a null `UNIT_TYPE` as well, likely due to incomplete reporting.

**Census EDA**

We check for missing values and address the `-666666666` placeholder values we noticed during data acquisition.

In [13]:
census_df = pd.read_csv('census.csv')

In [14]:
census_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1332 entries, 0 to 1331
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   median_household_income   1332 non-null   int64 
 1   total_population          1332 non-null   int64 
 2   race_universe_population  1332 non-null   int64 
 3   nh_white_population       1332 non-null   int64 
 4   total_households          1332 non-null   int64 
 5   zero_vehicle_households   1332 non-null   int64 
 6   NAME                      1332 non-null   object
 7   state                     1332 non-null   int64 
 8   county                    1332 non-null   int64 
 9   tract                     1332 non-null   int64 
dtypes: int64(9), object(1)
memory usage: 104.2+ KB


In [15]:
census_df['median_household_income'].describe()

count    1.332000e+03
mean    -7.422412e+06
std      7.038211e+07
min     -6.666667e+08
25%      5.580175e+04
50%      7.733200e+04
75%      1.057242e+05
max      2.500010e+05
Name: median_household_income, dtype: float64

In [16]:
(census_df['median_household_income'] == -666666666).sum()

np.int64(15)

The summary statistics are distorted by the 15 tracts with the value `-666666666`, which the Census Bureau uses as a placeholder for missing or unavailable data ([source](https://www.census.gov/content/dam/Census/library/publications/2023/acs/acs_table_based_summary_file_handbook_ch03.pdf)). We replace these with NaN and re-examine.

In [17]:
census_df['median_household_income'] = census_df['median_household_income'].replace(-666666666, np.nan)
census_df['median_household_income'].describe()

count      1317.000000
mean      86065.025057
std       42841.801332
min       13489.000000
25%       56460.000000
50%       78064.000000
75%      105982.000000
max      250001.000000
Name: median_household_income, dtype: float64

After replacing the missing data placeholders, 1,317 tracts have valid income data. Median household income across Cook County tracts is $78,000, ranging from $13,500 to the Census reporting cap of $250,001.

A small number of tracts have `total_population = 0`. These are special-purpose tracts (parks, airports, large institutional areas) where nobody lives. Because they cannot contribute to any per-capita rate or percentage calculation, we drop them from the Census dataset.

In [19]:
census_df[census_df['total_population'] == 0]

,median_household_income,total_population,race_universe_population,nh_white_population,total_households,zero_vehicle_households,NAME,state,county,tract
411,NaN,0,0,0,0,0,Census Tract 3817; Cook County; Illinois,17,31,381700
1329,NaN,0,0,0,0,0,Census Tract 9800; Cook County; Illinois,17,31,980000
1330,NaN,0,0,0,0,0,Census Tract 9801; Cook County; Illinois,17,31,980100
1331,NaN,0,0,0,0,0,Census Tract 9900; Cook County; Illinois,17,31,990000


In [20]:
zero_pop_tracts = census_df[census_df['total_population'] == 0]
print(f"Dropping {len(zero_pop_tracts)} zero-population tracts:")
print(zero_pop_tracts[['NAME', 'tract']].to_string(index=False))

Dropping 4 zero-population tracts:
                                    NAME  tract
Census Tract 3817; Cook County; Illinois 381700
Census Tract 9800; Cook County; Illinois 980000
Census Tract 9801; Cook County; Illinois 980100
Census Tract 9900; Cook County; Illinois 990000


In [21]:
census_df = census_df[census_df['total_population'] > 0].reset_index(drop=True)
print(f"\nRemaining tracts: {len(census_df)}")


Remaining tracts: 1328


We derive two percentage variables for the equity analysis: percent people of color and percent zero-vehicle households.

In [22]:
census_df['pct_poc'] = 1 - (census_df['nh_white_population'] / census_df['race_universe_population'])
census_df['pct_zero_vehicle_hh'] = census_df['zero_vehicle_households'] / census_df['total_households']

census_df[['pct_poc', 'pct_zero_vehicle_hh']].describe()

,pct_poc,pct_zero_vehicle_hh
count,1328.000000,1328.000000
mean,0.621853,0.179167
std,0.293120,0.150833
min,0.063537,0.000000
25%,0.359081,0.059092
50%,0.621930,0.134391
75%,0.927916,0.264024
max,1.000000,0.746725


In [24]:
census_df.to_csv('census.csv', index=False)

### EDA summary
EDA confirmed that all datasets are clean enough to proceed with analysis. Key findings:
- `CRASH_RECORD_ID` has no nulls across all three crash tables, ensuring reliable joins between them in the next phase.
- About 8,100 crash records lack valid coordinates (~0.8%) and will need to be excluded from spatial analysis.
- Crash data coverage is consistent from 2018-2025; earlier years are incomplete.
- The majority of crashes have no injury; fatal crashes account for ~0.1% of records.
- Null `VEHICLE_ID` values in the Vehicles table are explained by pedestrian and cyclist units.
- 15 Census tracts lack income data due to insufficient sample size.

## Part III: Analysis plan

### Spatial join: crashes to Census tracts

Our complex data analysis involves spatially joining crash coordinates to Census tract boundaries to attach median household income data to each crash. This enables us to analyze whether crash severity is disproportionately concentrated in lower-income neighborhoods.

In [36]:
import geopandas as gpd
from shapely.geometry import Point

We need Census tract boundary polygons to spatially join crash coordinates to tracts. We download the TIGER/Line shapefile for Illinois census tracts from the Census Bureau.

In [37]:
tracts_url = "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_17_tract.zip"
tracts_gdf = gpd.read_file(tracts_url)

In [38]:
tracts_gdf

,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,17,161,022800,17161022800,1400000US17161022800,228,Census Tract 228,G5020,S,2103943,0,+41.4991066,-090.5472913,"POLYGON ((-90.55724 41.49433, -90.55724 41.494..."
1,17,161,022900,17161022900,1400000US17161022900,229,Census Tract 229,G5020,S,1311568,0,+41.4897868,-090.5477824,"POLYGON ((-90.55731 41.49337, -90.55727 41.493..."
2,17,161,023000,17161023000,1400000US17161023000,230,Census Tract 230,G5020,S,4680875,127822,+41.4757633,-090.5435406,"POLYGON ((-90.55779 41.46279, -90.55772 41.463..."
3,17,161,023100,17161023100,1400000US17161023100,231,Census Tract 231,G5020,S,3496624,66115,+41.4684205,-090.5721252,"POLYGON ((-90.5826 41.46694, -90.5826 41.46714..."
4,17,019,010901,17019010901,1400000US17019010901,109.01,Census Tract 109.01,G5020,S,140475117,118829,+39.9889767,-088.2028522,"POLYGON ((-88.27831 39.88075, -88.27786 39.883..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3260,17,195,001300,17195001300,1400000US17195001300,13,Census Tract 13,G5020,S,3058885,0,+41.8026590,-089.6673228,"POLYGON ((-89.67947 41.79957, -89.67946 41.800..."
3261,17,195,000600,17195000600,1400000US17195000600,6,Census Tract 6,G5020,S,352671538,4367544,+41.7023883,-090.0729869,"POLYGON ((-90.24791 41.74694, -90.2479 41.7470..."
3262,17,195,000800,17195000800,1400000US17195000800,8,Census Tract 8,G5020,S,331559611,1574583,+41.6502472,-089.7773192,"POLYGON ((-89.92104 41.59391, -89.92093 41.593..."
3263,17,089,850400,17089850400,1400000US17089850400,8504,Census Tract 8504,G5020,S,4478955,609174,+42.0974370,-088.2480785,"POLYGON ((-88.28454 42.10354, -88.28437 42.103..."


The shapefile contains all Illinois tracts. We filter to Cook County (FIPS code 031) since our crash data is within Chicago.

In [39]:
cook_tracts = tracts_gdf[tracts_gdf['COUNTYFP'] == '031']

In [40]:
cook_tracts

,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
173,17,031,807100,17031807100,1400000US17031807100,8071,Census Tract 8071,G5020,S,1528033,0,+42.0444535,-087.7210786,"POLYGON ((-87.73261 42.04094, -87.7326 42.0412..."
198,17,031,030101,17031030101,1400000US17031030101,301.01,Census Tract 301.01,G5020,S,127283,110999,+41.9910538,-087.6548790,"POLYGON ((-87.66034 41.99229, -87.66011 41.992..."
199,17,031,030701,17031030701,1400000US17031030701,307.01,Census Tract 307.01,G5020,S,69878,0,+41.9827453,-087.6579659,"POLYGON ((-87.66007 41.98362, -87.65984 41.983..."
200,17,031,070103,17031070103,1400000US17031070103,701.03,Census Tract 701.03,G5020,S,69094,0,+41.9268110,-087.6426051,"POLYGON ((-87.64422 41.92805, -87.64381 41.928..."
201,17,031,807200,17031807200,1400000US17031807200,8072,Census Tract 8072,G5020,S,2489125,0,+42.0371751,-087.7285103,"POLYGON ((-87.74729 42.03369, -87.74728 42.033..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066,17,031,804607,17031804607,1400000US17031804607,8046.07,Census Tract 8046.07,G5020,S,3639630,7556,+41.9973360,-088.0520065,"POLYGON ((-88.08023 41.99221, -88.08022 41.992..."
3068,17,031,804709,17031804709,1400000US17031804709,8047.09,Census Tract 8047.09,G5020,S,3045418,30138,+42.0374913,-088.0874060,"POLYGON ((-88.10197 42.03849, -88.10196 42.038..."
3069,17,031,826800,17031826800,1400000US17031826800,8268,Census Tract 8268,G5020,S,7943308,0,+41.6373647,-087.6600155,"POLYGON ((-87.67975 41.64819, -87.6791 41.6482..."
3070,17,031,710300,17031710300,1400000US17031710300,7103,Census Tract 7103,G5020,S,327581,0,+41.7540485,-087.6611057,"POLYGON ((-87.66361 41.75768, -87.6633 41.7576..."


The 1,332 tracts match our Census income data. We now convert the crash coordinates into a GeoDataFrame for the spatial join.

In [41]:
crashes_geo = crashes_df.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()
crashes_geo = gpd.GeoDataFrame(
    crashes_geo,
    geometry=gpd.points_from_xy(crashes_geo['LONGITUDE'], crashes_geo['LATITUDE']),
    crs="EPSG:4326"
)

crashes_geo

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION,geometry
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,NaN,2026-04-18 01:45:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,NOT DIVIDED,...,0.0,3.0,0.0,1,7,4,41.764609,-87.683387,POINT (-87.683387053281 41.764608912236),POINT (-87.68339 41.76461)
1,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,NaN,2026-04-18 01:17:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,...,0.0,6.0,0.0,1,7,4,41.886708,-87.629602,POINT (-87.629601563147 41.886708085892),POINT (-87.6296 41.88671)
2,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,NaN,2026-04-18 01:04:00,20,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,1.0,0.0,1,7,4,41.969284,-87.718427,POINT (-87.718426836301 41.969284284638),POINT (-87.71843 41.96928)
3,55e705a664cbd0ad3961d701adc8af91e07b95904ea670...,NaN,2026-04-18 00:10:00,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,0.0,4.0,0.0,0,7,4,41.943223,-87.747131,POINT (-87.747130750615 41.943222858294),POINT (-87.74713 41.94322)
4,b4090532587c955b92c35d9475fac5a9aaa2bb5c6d7e27...,NaN,2026-04-17 23:50:00,30,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,DIVIDED - W/MEDIAN BARRIER,...,0.0,2.0,0.0,23,6,4,41.887056,-87.634937,POINT (-87.634937037709 41.887056468859),POINT (-87.63494 41.88706)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046564,1d0232afecbdfd01968555aa956a688fd6f55a2bd1984f...,NaN,2014-02-24 19:45:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,DAYLIGHT,TURNING,NOT DIVIDED,...,0.0,2.0,0.0,19,2,2,41.884016,-87.701143,POINT (-87.701142757538 41.884016475152),POINT (-87.70114 41.88402)
1046565,957783a4787318f005a7dbc920e4c84cb9ac8aa7329a62...,NaN,2014-01-21 07:40:00,30,YIELD,NO CONTROLS,CLEAR,DAYLIGHT,ANGLE,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,1.0,0.0,7,3,1,41.760710,-87.561946,POINT (-87.561946030143 41.760710194223),POINT (-87.56195 41.76071)
1046566,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,NaN,2014-01-18 18:14:00,30,NO CONTROLS,NO CONTROLS,CLEAR,DUSK,PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN BARRIER,...,0.0,2.0,0.0,18,7,1,41.885610,-87.638756,POINT (-87.638756189808 41.885609917063),POINT (-87.63876 41.88561)
1046567,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,2013-06-01 20:29:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PEDESTRIAN,NOT DIVIDED,...,0.0,1.0,0.0,20,7,6,42.005145,-87.689969,POINT (-87.689968732521 42.005144534455),POINT (-87.68997 42.00514)


The GeoDataFrame has 1,038,468 rows, down from 1,046,569 in the original Crashes dataset. The difference of 8,101 corresponds to the records with missing or invalid coordinates (8,030 original nulls + 71 zero-value coordinates we replaced with NaN).

We now spatially join each crash point to the Census tract polygon it falls within.

In [42]:
cook_tracts = cook_tracts.to_crs("EPSG:4326")

In [43]:
crashes_with_tract = gpd.sjoin(crashes_geo, cook_tracts, how='left', predicate='within')

In [44]:
crashes_with_tract

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,NaN,2026-04-18 01:45:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,NOT DIVIDED,...,17031661000,1400000US17031661000,6610,Census Tract 6610,G5020,S,1118966.0,0.0,+41.7618177,-087.6850469
1,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,NaN,2026-04-18 01:17:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,...,17031839100,1400000US17031839100,8391,Census Tract 8391,G5020,S,1101771.0,84111.0,+41.8809857,-087.6330703
2,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,NaN,2026-04-18 01:04:00,20,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,17031140301,1400000US17031140301,1403.01,Census Tract 1403.01,G5020,S,164129.0,0.0,+41.9692055,-087.7231695
3,55e705a664cbd0ad3961d701adc8af91e07b95904ea670...,NaN,2026-04-18 00:10:00,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,17031151002,1400000US17031151002,1510.02,Census Tract 1510.02,G5020,S,637151.0,0.0,+41.9424917,-087.7471554
4,b4090532587c955b92c35d9475fac5a9aaa2bb5c6d7e27...,NaN,2026-04-17 23:50:00,30,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,DIVIDED - W/MEDIAN BARRIER,...,17031839100,1400000US17031839100,8391,Census Tract 8391,G5020,S,1101771.0,84111.0,+41.8809857,-087.6330703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046564,1d0232afecbdfd01968555aa956a688fd6f55a2bd1984f...,NaN,2014-02-24 19:45:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,DAYLIGHT,TURNING,NOT DIVIDED,...,17031837100,1400000US17031837100,8371,Census Tract 8371,G5020,S,745370.0,0.0,+41.8856736,-087.6987204
1046565,957783a4787318f005a7dbc920e4c84cb9ac8aa7329a62...,NaN,2014-01-21 07:40:00,30,YIELD,NO CONTROLS,CLEAR,DAYLIGHT,ANGLE,DIVIDED - W/MEDIAN (NOT RAISED),...,17031431302,1400000US17031431302,4313.02,Census Tract 4313.02,G5020,S,437419.0,0.0,+41.7583855,-087.5627859
1046566,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,NaN,2014-01-18 18:14:00,30,NO CONTROLS,NO CONTROLS,CLEAR,DUSK,PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN BARRIER,...,17031280100,1400000US17031280100,2801,Census Tract 2801,G5020,S,581716.0,29634.0,+41.8854520,-087.6434977
1046567,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,2013-06-01 20:29:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PEDESTRIAN,NOT DIVIDED,...,17031020400,1400000US17031020400,204,Census Tract 204,G5020,S,555577.0,0.0,+42.0084880,-087.6859118


The number of rows remain the same after the spatial join (1,038,468). This confirms no duplicate matches were created. The join added 14 columns from the tract shapefile, bringing the total from 49 to 63. The key column is `TRACTCE`, which identifies the Census tract each crash falls within.

In [45]:
crashes_with_tract['TRACTCE'].head(10)

0    661000
1    839100
2    140301
3    151002
4    839100
5    081300
6    430700
7    230700
8    210900
9    081600
Name: TRACTCE, dtype: object

In [46]:
census_df['tract'].head(10)

0    10100
1    10201
2    10202
3    10300
4    10400
5    10501
6    10502
7    10503
8    10600
9    10701
Name: tract, dtype: int64

The tract codes from the spatial join (`TRACTCE`) are 6-character strings (e.g. `661000`, `081300`), while the Census income data stores them as shorter integers (e.g. `10100`). We convert the Census tract codes to 6-character strings with leading zeros to align the formats for merging.

In [47]:
census_df['tract'] = census_df['tract'].astype(str).str.zfill(6)
census_df['tract'].head()

0    010100
1    010201
2    010202
3    010300
4    010400
Name: tract, dtype: object

Checking how many crashes did not match to a Cook County tract.

In [48]:
crashes_with_tract['TRACTCE'].isnull().sum()

np.int64(89)

In [49]:
crashes_with_tract[crashes_with_tract['TRACTCE'].isnull()][['LATITUDE', 'LONGITUDE']].describe()

,LATITUDE,LONGITUDE
count,89.000000,89.000000
mean,41.958382,-87.932447
std,0.003548,0.003152
min,41.955309,-87.939678
25%,41.956556,-87.933994
50%,41.958987,-87.933994
75%,41.958987,-87.933028
max,41.987827,-87.923620


The 89 unmatched crashes have valid Chicago coordinates (41.96, -87.93) but fall just outside Cook County tract boundaries. This is a negligible fraction of the data.

We now merge the Census median household income data onto the spatially joined crash-tract dataset using the aligned tract codes.

In [50]:
crashes_with_income = crashes_with_tract.merge(
    census_df[['tract', 'median_household_income']],
    left_on='TRACTCE',
    right_on='tract',
    how='left'
)

In [51]:
crashes_with_income

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,tract,median_household_income
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,NaN,2026-04-18 01:45:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",REAR END,NOT DIVIDED,...,6610,Census Tract 6610,G5020,S,1118966.0,0.0,+41.7618177,-087.6850469,661000,49652.0
1,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,NaN,2026-04-18 01:17:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,...,8391,Census Tract 8391,G5020,S,1101771.0,84111.0,+41.8809857,-087.6330703,839100,139943.0
2,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,NaN,2026-04-18 01:04:00,20,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN (NOT RAISED),...,1403.01,Census Tract 1403.01,G5020,S,164129.0,0.0,+41.9692055,-087.7231695,140301,71648.0
3,55e705a664cbd0ad3961d701adc8af91e07b95904ea670...,NaN,2026-04-18 00:10:00,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,RAIN,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,1510.02,Census Tract 1510.02,G5020,S,637151.0,0.0,+41.9424917,-087.7471554,151002,78851.0
4,b4090532587c955b92c35d9475fac5a9aaa2bb5c6d7e27...,NaN,2026-04-17 23:50:00,30,NO CONTROLS,NO CONTROLS,RAIN,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,DIVIDED - W/MEDIAN BARRIER,...,8391,Census Tract 8391,G5020,S,1101771.0,84111.0,+41.8809857,-087.6330703,839100,139943.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038463,1d0232afecbdfd01968555aa956a688fd6f55a2bd1984f...,NaN,2014-02-24 19:45:00,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,RAIN,DAYLIGHT,TURNING,NOT DIVIDED,...,8371,Census Tract 8371,G5020,S,745370.0,0.0,+41.8856736,-087.6987204,837100,61351.0
1038464,957783a4787318f005a7dbc920e4c84cb9ac8aa7329a62...,NaN,2014-01-21 07:40:00,30,YIELD,NO CONTROLS,CLEAR,DAYLIGHT,ANGLE,DIVIDED - W/MEDIAN (NOT RAISED),...,4313.02,Census Tract 4313.02,G5020,S,437419.0,0.0,+41.7583855,-087.5627859,431302,31981.0
1038465,f62e27317feb174811cf4fefeb9fa1064fea6c0619a873...,NaN,2014-01-18 18:14:00,30,NO CONTROLS,NO CONTROLS,CLEAR,DUSK,PARKED MOTOR VEHICLE,DIVIDED - W/MEDIAN BARRIER,...,2801,Census Tract 2801,G5020,S,581716.0,29634.0,+41.8854520,-087.6434977,280100,148319.0
1038466,19fb5af681f833c2af85734245f737fa6fbe62ac1ea379...,NaN,2013-06-01 20:29:00,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PEDESTRIAN,NOT DIVIDED,...,204,Census Tract 204,G5020,S,555577.0,0.0,+42.0084880,-087.6859118,020400,81840.0


In [52]:
len(crashes_with_income)

1038468

The row count remains the same after the merge (1,038,468). This confirms a clean one-to-one join where each crash matched to at most one tract's income value. Two new columns were added: `tract` and `median_household_income`.

In [53]:
crashes_with_income[['CRASH_RECORD_ID', 'TRACTCE', 'median_household_income']].head(10)

,CRASH_RECORD_ID,TRACTCE,median_household_income
0,c85240873b1b2563cb5e0c90e06afa99b74bb758cfa97d...,661000,49652.0
1,e4ab2f4a37a019fd8492c3c68305d7c48ac0f5a7c9d58d...,839100,139943.0
2,b2c5a8984887782f45655931ea07ea1699215b7446ce04...,140301,71648.0
3,55e705a664cbd0ad3961d701adc8af91e07b95904ea670...,151002,78851.0
4,b4090532587c955b92c35d9475fac5a9aaa2bb5c6d7e27...,839100,139943.0
5,d0c02b127af5eba533ec0ea76fc95f461669ba85df42e8...,081300,115260.0
6,3c5b647449be4f1a58d8329f859a5ff53d597bc04f37b2...,430700,37983.0
7,38e99ae39d27bc15c1d0b94a186d24d15eaa5452188bbb...,230700,33994.0
8,c6f16d0eef72c627b5e699606fc9b9e1ee47dadaf973bd...,210900,106083.0
9,4f9da2b2cc48d7e9b8dd32474c6748b77ea4676d73cdea...,081600,140323.0


In [142]:
crashes_with_income.to_csv('crashes_with_income.csv', index=False)

**the spatial join Summary**

We successfully performed our core complex data analysis by spatially joining crash coordinates to Census tract boundaries. This involved:
- Downloading Census tract boundary shapefiles for Illinois and filtering to Cook County (1,332 tracts)
- Converting crash lat/lon coordinates into a GeoDataFrame
- Spatially joining each crash point to the Census tract polygon it falls within
- Aligning tract code formats between the shapefile and Census income data
- Merging median household income onto each crash record

Of the 1,038,468 crashes with valid coordinates, only 89 (0.009%) did not match a Cook County tract. The final dataset links each crash to the median household income of the neighborhood where it occurred. This forms the foundation for our equity analysis.

### Validation

We validate the spatial join by sampling crash records and manually verifying that each crash was assigned the correct Census tract using the [Census Bureau Geocoder](https://geocoding.geo.census.gov/geocoder/geographies/coordinates).

In [54]:
sample = crashes_with_income[['LATITUDE', 'LONGITUDE', 'TRACTCE', 'median_household_income']].dropna().sample(10, random_state=33)

sample


,LATITUDE,LONGITUDE,TRACTCE,median_household_income
159361,41.745172,-87.682584,700501,64739.0
138248,41.763334,-87.630808,691100,55170.0
142279,41.880415,-87.742950,252202,45839.0
63249,41.912488,-87.765475,250300,64537.0
292531,41.744080,-87.604974,440102,36650.0
934421,41.856969,-87.722553,292400,40152.0
652247,41.874454,-87.647224,841900,136955.0
955589,41.721103,-87.666688,720200,121080.0
714318,41.906244,-87.669862,241500,178750.0
1016529,41.890823,-87.624214,081403,117550.0


We looked up each of the 10 coordinates on the Census Bureau Geocoder and confirmed that the returned tract matched our assigned `TRACTCE` in every case. The spatial join is producing accurate results.

### Community area boundaries

Our analysis aggregates findings to the level of Chicago's 77 community areas which is the geographic unit the city uses for policy and resource allocation. We download the official Community Areas boundary file from the Chicago Data Portal (`cauq-8yn6`) for use in spatially joining crashes to community areas and for mapping.

Source: https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-current-/cauq-8yn6

We test the connection by loading the boundaries directly from the Chicago Data Portal's GeoJSON export endpoint. We download the GeoJSON file directly from the Chicago Data Portal's geospatial export endpoint and save it locally before reading it with `geopandas`.

In [26]:
import geopandas as gpd

In [33]:
ca_url = "https://data.cityofchicago.org/resource/igwz-8jzy.geojson?$limit=100"
response = requests.get(ca_url)
print(response.status_code)
print(f"Response size: {len(response.content):,} bytes")

200
Response size: 2,072,102 bytes


In [34]:
with open('community_areas.geojson', 'wb') as f:
    f.write(response.content)

In [35]:
community_areas = gpd.read_file('community_areas.geojson')
print(community_areas.shape)

(77, 6)


In [36]:
community_areas.columns.tolist()

['community',
 'shape_area',
 'area_num_1',
 'area_numbe',
 'shape_len',
 'geometry']

In [37]:
community_areas

,community,shape_area,area_num_1,area_numbe,shape_len,geometry
0,ROGERS PARK,51259902.4506,1,1,34052.3975757,"MULTIPOLYGON (((-87.65456 41.99817, -87.65574 ..."
1,WEST RIDGE,98429094.8621,2,2,43020.6894583,"MULTIPOLYGON (((-87.68465 42.01948, -87.68464 ..."
2,UPTOWN,65095642.7289,3,3,46972.7945549,"MULTIPOLYGON (((-87.64102 41.9548, -87.644 41...."
3,LINCOLN SQUARE,71352328.2399,4,4,36624.6030848,"MULTIPOLYGON (((-87.67441 41.9761, -87.6744 41..."
4,NORTH CENTER,57054167.85,5,5,31391.6697542,"MULTIPOLYGON (((-87.67336 41.93234, -87.67342 ..."
...,...,...,...,...,...,...
72,WASHINGTON HEIGHTS,79635752.8769,73,73,42222.598163,"MULTIPOLYGON (((-87.63373 41.72885, -87.63369 ..."
73,MOUNT GREENWOOD,75584290.0209,74,74,48665.1305392,"MULTIPOLYGON (((-87.69646 41.70714, -87.69644 ..."
74,MORGAN PARK,91877340.6988,75,75,46396.419362,"MULTIPOLYGON (((-87.64215 41.68508, -87.64249 ..."
75,OHARE,371835607.687,76,76,173625.98466,"MULTIPOLYGON (((-87.83658 41.9864, -87.83658 4..."


In [38]:
community_areas.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [39]:
community_areas.geometry.isna().sum()

np.int64(0)

#### Summary
We successfully acquired the Chicago Community Areas boundary file via the Chicago Data Portal SODA API (resource `igwz-8jzy`):
- **77 community area polygons**, matching the city's official count
- CRS is **EPSG:4326 (WGS 84)**, aligning with our crash GeoDataFrame for spatial joins
- No null geometries
- Saved locally as `community_areas.geojson`

This file will be used in the next phase to spatially join each crash to its community area and to aggregate Census tract demographics up to the community area level for our equity analysis.